# 🏈 Fantasy Football Draft Analysis (2024–2025)
*Justin McKendry · August 2025*

---

## 📚 Project Overview

This notebook analyzes player performance, draft value, and team efficiency based on our league's draft and season data.  
We explore who maximized their draft capital, which picks over- or under-performed, and how VORP (Value Over Replacement Player) relates to league standings.

**Key Metrics:**
- 📈 VORP (custom-calculated per position)
- 🎯 Draft Delta (Actual Pick - ADP)
- 💥 Boom/Bust Score (Actual - Projected points)

---

## 🧪 1. Data Collection

### 🔍 Purpose
Connect to ESPN's Fantasy API and extract:
- League metadata
- Player season stats
- Draft results
- Final standings

Fantasy Pros:
https://www.fantasypros.com/nfl/adp/overall.php
Get a JSON file from fantasy pros to find the average draft position of players. 


In [ ]:
import pandas as pd 

import sys, os

from utils import get_league_data, get_all_player_stats, save_to_data_raw, process_projections
from config import CURRENT_SEASON, NEXT_SEASON


### Loading League Data from ESPN API

Trying to load all league data for the past 


In [ ]:
leagues = [get_league_data(year) for year in range(2020,CURRENT_SEASON+1)]


In [ ]:
leagues

## Creating and Saving Player Dataframe

### Calling the get_all_player_stats 
Call the function to get all the NFL players in the league into one dataframe that contains their stats across the course of the entire season

In [ ]:
df_players = pd.concat(
    [pd.DataFrame(get_all_player_stats(league, num_fa=300))
              .assign(year=league.year)
              for league in leagues]) # Collects the player stats by calling the get_all_player_stats function



In [ ]:
df_players = pd.concat(
    [pd.DataFrame(get_all_player_stats(league, num_fa=300))
              .assign(year=league.year)
              for league in leagues]) # Collects the player stats by calling the get_all_player_stats function


In [ ]:
# Next-season projections are pulled by `notebooks/pull_projections.py`, not here.
#
# This cell used to read:
#     curr_players = curr_season.free_agents(size=500)
#
# which is only correct while the league sits in its pre-draft state. Run at
# any other time, `free_agents()` structurally excludes every rostered player -
# i.e. exactly the elite tier. The 2025 pull was made mid-season, so the file it
# produced contained none of the top 50 players by ADP, and the draft board could
# only ever recommend from the leftovers.
#
# The script unions free agents with every team's roster, so it stays correct
# whenever it happens to be run:
#
#     python notebooks/pull_projections.py --year 2026
#     python notebooks/rebuild_projections.py --year 2026
#
# The second script is NB02's projection step on its own, so refreshing
# projections doesn't require re-running the whole notebook (whose ADP insert is
# explicitly not safe to repeat).


In [ ]:
# draft = curr_season.draft
# draft

In [ ]:
player = leagues[0].teams[0].roster[0]
stats = player.stats  # This returns a list of stat dicts, usually per week
player.stats.keys()
process_projections

In [ ]:
for week in range(1, 18):
    box_scores = leagues[0].box_scores(week)
    for box in box_scores:
        for player in box.home_lineup + box.away_lineup:
            print(f"Week {week}: {player.name} - {player.points}")

In [ ]:
df_players[['eligible_slots', 'player_name','year','acquisition_type']]

### Saving to a CSV
Calls the save_to_data_raw function to save the dataframe in a file within the data/raw folder

In [ ]:
df_players

In [ ]:
[save_to_data_raw(df_players[df_players['year'] == league.year], 
                  'player_stats', league.year) 
 for league in leagues]

## Creating and Saving League Specific Draft dataframe

### Getting the data
Calling the espn_request.get_league_draft function to get a json of all league specific draft data from 2024. Save this as a dataframe called df_draft. 

In [ ]:
# ESPN's raw picks carry `memberId` - the drafter's SWID, which alone is enough
# to enumerate every league that person is in. Dropped here so it never reaches
# the frame, let alone the CSV. Ownership downstream is keyed on `team_id`,
# which is also the only thing that works: autodrafted picks have no memberId.
drafts = [
    pd.DataFrame(league.espn_request.get_league_draft()['draftDetail']['picks'])
    .drop(columns=['memberId'], errors='ignore')
    .assign(year=league.year, league_id=league.league_id)
    for league in leagues
]
df_all_drafts = pd.concat(drafts, ignore_index=True)
df_all_drafts

### Cleaning the data
There are multiple columns that are empty as those settings are not turned on in our league. Need to rename some of the columns as well to be able to communicate across tables in the database

In [ ]:
# Dropping irrelevant columns
df_all_drafts = df_all_drafts.drop(columns=['bidAmount', 'keeper', 'nominatingTeamId', 'reservedForKeeper', 'tradeLocked'])

# Renaming columns to database convention
df_all_drafts = df_all_drafts.rename(columns={'playerId': 'player_id', 'teamId': 'team_id'})
df_all_drafts

### Saving to a csv
Call the save_to_data_raw function again to save the draft data as a csv in data/raw folder

In [ ]:
[save_to_data_raw(df_all_drafts[df_all_drafts['year'] == league.year], 
                  'draft_data', league.year) 
 for league in leagues]

## Getting League Specific Team Data

### Getting the data into a dataframe
Loop through a list of league.teams. This is a list of fantasy football teams created within our league. It contains data like wins and losses as well as points_for and points_against. The draft_projected_rank and final_standing are interesting as well.

In [ ]:
df_teams = pd.DataFrame([
    {
        "team_id": team.team_id,
        "team_name": team.team_name,
        "abbrev": team.team_abbrev,
        "division_id": team.division_id,
        "division_name": team.division_name,
        "wins": team.wins,
        "losses": team.losses,
        "ties": team.ties,
        "points_for": team.points_for,
        "points_against": team.points_against,
        "draft_projected_rank": team.draft_projected_rank,
        "final_standing": team.final_standing,
        "year": league.year
    }
    for league in leagues
    for team in league.teams
])
df_teams


### Saving to a csv

In [ ]:
[save_to_data_raw(df_teams[df_teams['year'] == league.year], 
                  'teams_data', league.year) 
 for league in leagues]